# 🧬 BBBP Penetration Prediction — End-to-End ML Project

**Goal:** predict binary blood-brain barrier penetration from molecular structure.

Pipeline:

**SMILES → RDKit descriptors + Morgan fingerprints → scaffold split → model comparison → evaluation → single-molecule prediction**

> This is a benchmark ML project, not a clinical decision tool.

In [ ]:
# Cell 1 — Install dependencies
!pip -q install rdkit scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
# Cell 2 — Imports
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
FP_SIZE = 256

In [ ]:
# Cell 3 — Download the MoleculeNet BBBP dataset
DATA_URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(DATA_URL)

print("Shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())

In [ ]:
# Cell 4 — Basic data audit
print("Missing values:")
display(df.isna().sum())

print("\nTarget distribution:")
display(df["p_np"].value_counts(dropna=False).rename(index={0:"BBB-", 1:"BBB+"}))

print("\nDuplicate SMILES:", df["smiles"].duplicated().sum())

In [ ]:
# Cell 5 — Clean the dataset
df = df.dropna(subset=["smiles", "p_np"]).copy()
df = df.drop_duplicates(subset=["smiles"]).reset_index(drop=True)

def parse_smiles(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    return Chem.MolFromSmiles(smiles)

df["mol"] = df["smiles"].map(parse_smiles)
invalid = df["mol"].isna().sum()
print("Invalid SMILES:", invalid)

df = df[df["mol"].notna()].reset_index(drop=True)
print("Clean dataset shape:", df.shape)

In [ ]:
# Cell 6 — Molecular descriptor extraction
def descriptor_dict(mol):
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
    }

desc_df = pd.DataFrame([descriptor_dict(m) for m in df["mol"]])
display(desc_df.describe().T)

In [ ]:
# Cell 7 — Morgan fingerprints + combined feature matrix
def morgan_bits(mol, n_bits=FP_SIZE):
    fp = rdMolDescriptors.GetMorganGenerator(radius=2, fpSize=n_bits).GetFingerprint(mol)
    arr = np.zeros((n_bits,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fingerprints = np.vstack([morgan_bits(m) for m in df["mol"]])

X = np.hstack([
    desc_df.to_numpy(dtype=float),
    fingerprints.astype(float)
])
y = df["p_np"].astype(int).to_numpy()

print("Descriptor features:", desc_df.shape[1])
print("Fingerprint features:", fingerprints.shape[1])
print("Total features:", X.shape[1])

In [ ]:
# Cell 8 — Optional EDA: class balance
counts = df["p_np"].value_counts().sort_index()
plt.figure(figsize=(6,4))
plt.bar(["BBB-", "BBB+"], counts.values)
plt.ylabel("Molecules")
plt.title("BBBP Class Distribution")
plt.show()

In [ ]:
# Cell 9 — Scaffold split
def scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)

df["_scaffold"] = df["smiles"].map(scaffold)

rng = np.random.default_rng(RANDOM_STATE)
groups = list(df.groupby("_scaffold").groups.values())
rng.shuffle(groups)

target_test = int(np.ceil(len(df) * 0.20))
test_idx = []
n = 0
for group in groups:
    if n >= target_test:
        break
    test_idx.extend(list(group))
    n += len(group)

test_idx = set(test_idx)
train_idx = [i for i in df.index if i not in test_idx]

X_train, X_test = X[train_idx], X[list(test_idx)]
y_train, y_test = y[train_idx], y[list(test_idx)]

print("Train:", len(train_idx))
print("Test:", len(test_idx))
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())

In [ ]:
# Cell 10 — Train multiple models
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=500, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=500, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }

results_df = pd.DataFrame(results).T.sort_values("ROC-AUC", ascending=False)
display(results_df)

In [ ]:
# Cell 11 — Detailed evaluation of best model
best_name = results_df.index[0]
best_model = models[best_name]

pred = best_model.predict(X_test)
proba = best_model.predict_proba(X_test)[:, 1]

print("Best model:", best_name)
print("\nClassification report:")
print(classification_report(y_test, pred, target_names=["BBB-", "BBB+"]))

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["BBB-", "BBB+"],
            yticklabels=["BBB-", "BBB+"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {best_name}")
plt.show()

In [ ]:
# Cell 12 — ROC curve
RocCurveDisplay.from_predictions(y_test, proba)
plt.title(f"ROC Curve — {best_name}")
plt.show()

In [ ]:
# Cell 13 — Descriptor-level exploratory interpretation
corr = pd.concat([desc_df, df["p_np"].rename("BBB_positive")], axis=1).corr(numeric_only=True)["BBB_positive"].drop("BBB_positive")
display(corr.sort_values())

In [ ]:
# Cell 14 — Single-molecule prediction
def predict_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES.")

    desc = descriptor_dict(mol)
    fp = morgan_bits(mol)
    vector = np.concatenate([np.array(list(desc.values()), dtype=float), fp.astype(float)])

    p = float(best_model.predict_proba([vector])[0, 1])
    label = "BBB+" if p >= 0.5 else "BBB-"

    return {
        "prediction": label,
        "BBB+ probability": p,
        "descriptors": desc,
        "molecule": mol,
    }

example = predict_smiles("CCO")
print(example["prediction"])
print(f"BBB+ probability: {example['BBB+ probability']:.1%}")
display(pd.DataFrame([example["descriptors"]]))
display(Draw.MolToImage(example["molecule"], size=(500, 350)))

In [ ]:
# Cell 15 — Save model + metadata
import joblib, os

os.makedirs("models", exist_ok=True)
joblib.dump(best_model, "models/bbbp_model.joblib")

metadata = {
    "best_model": best_name,
    "feature_count": int(X.shape[1]),
    "fingerprint_size": FP_SIZE,
    "descriptors": list(desc_df.columns),
    "split": "scaffold",
    "test_size": 0.20,
    "random_state": RANDOM_STATE,
}
with open("models/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved model and metadata.")

In [ ]:
# Cell 16 — Create a clean prediction function for deployment
def deployed_predict(smiles):
    model = joblib.load("models/bbbp_model.joblib")
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES.")

    desc = descriptor_dict(mol)
    fp = morgan_bits(mol)
    vector = np.concatenate([np.array(list(desc.values()), dtype=float), fp.astype(float)])

    probability = float(model.predict_proba([vector])[0, 1])
    return {
        "class": "BBB+" if probability >= 0.5 else "BBB-",
        "probability": probability,
        "descriptors": desc
    }

deployed_predict("CCO")

In [ ]:
# Cell 17 — Download the trained model for GitHub/local deployment
from google.colab import files
files.download("models/bbbp_model.joblib")
files.download("models/metadata.json")